# Crypto Backtesting Starter Kit
### By JW Quant Lab

A complete pipeline: **fetch data → build strategy → measure performance → analyze costs.**

Most backtests you see online are **dangerously optimistic** because they ignore trading costs.
This notebook shows you exactly how much costs matter — and why 90% of "profitable" strategies fail in live trading.

**What you'll build:**
1. Fetch real BTC/USDT daily data from Binance (no API key needed)
2. Implement SMA(50/200) Golden Cross / Death Cross strategy
3. Calculate professional metrics: Sharpe, CAGR, Max Drawdown, Win Rate
4. Visualize equity curve and drawdowns
5. **Cost sensitivity analysis** — the one thing that separates real quants from dreamers

---
*Requirements: `pip install ccxt pandas numpy matplotlib`*

In [ ]:
import ccxt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

plt.style.use("dark_background")
plt.rcParams.update({"figure.figsize": (14, 5), "font.size": 11})

## Part 1: Fetch BTC/USDT Daily Data

Using `ccxt` to pull OHLCV data from Binance's **public API** — no API key required.
We fetch daily candles from 2020 onward to get enough history for SMA(200).

In [ ]:
exchange = ccxt.binance({"enableRateLimit": True})

# Fetch daily OHLCV — no API key needed for public data
since = exchange.parse8601("2020-01-01T00:00:00Z")
all_candles = []
while True:
    candles = exchange.fetch_ohlcv("BTC/USDT", "1d", since=since, limit=1000)
    if not candles:
        break
    all_candles.extend(candles)
    since = candles[-1][0] + 86400_000  # next day
    if len(candles) < 1000:
        break

df = pd.DataFrame(all_candles, columns=["timestamp", "open", "high", "low", "close", "volume"])
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
df.set_index("timestamp", inplace=True)
df = df[~df.index.duplicated(keep="first")]

print(f"Fetched {len(df)} daily candles: {df.index[0].date()} to {df.index[-1].date()}")
df.tail()

## Part 2: SMA(50/200) Crossover Strategy

**Rules:**
- **Buy** when SMA(50) crosses above SMA(200) — *Golden Cross*
- **Sell** when SMA(50) crosses below SMA(200) — *Death Cross*
- **T+1 execution**: signal on day N, execute on day N+1 open (realistic, no look-ahead bias)

In [ ]:
# Compute SMAs
df["sma_50"] = df["close"].rolling(50).mean()
df["sma_200"] = df["close"].rolling(200).mean()

# Signal: 1 when fast > slow, 0 otherwise
df["signal"] = (df["sma_50"] > df["sma_200"]).astype(int)

# T+1 execution: use YESTERDAY's signal for today's return
df["position"] = df["signal"].shift(1).fillna(0)

# Daily returns
df["ret"] = df["close"].pct_change()

# Strategy return = market return × position
df["strat_ret"] = df["ret"] * df["position"]

# Trade events (for cost calculation later)
df["trade"] = df["position"].diff().abs().fillna(0)

print(f"Total crossover trades: {int(df['trade'].sum())}")
print(f"Time in market: {df['position'].mean():.1%}")

# Quick plot: price + SMAs + signals
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["close"], color="#8b949e", linewidth=0.8, label="BTC/USDT")
ax.plot(df.index, df["sma_50"], color="#58a6ff", linewidth=1, label="SMA(50)")
ax.plot(df.index, df["sma_200"], color="#f0883e", linewidth=1, label="SMA(200)")

# Mark buy/sell signals
buys = df[df["trade"] == 1][df["position"] == 1]
sells = df[df["trade"] == 1][df["position"] == 0]
ax.scatter(buys.index, buys["close"], marker="^", color="#3fb950", s=100, zorder=5, label="Buy")
ax.scatter(sells.index, sells["close"], marker="v", color="#f85149", s=100, zorder=5, label="Sell")

ax.set_title("BTC/USDT — SMA(50/200) Crossover Signals", fontsize=14)
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
ax.set_facecolor("#0d1117")
fig.patch.set_facecolor("#0d1117")
plt.tight_layout()
plt.show()

## Part 3: Performance Metrics

Professional-grade metrics that every serious backtest should report.
If a backtest doesn't show these, **don't trust it.**

In [ ]:
def compute_metrics(returns: pd.Series, trades: int = 0, label: str = "") -> dict:
    """Compute backtest performance metrics from a daily return series."""
    equity = (1 + returns).cumprod()
    total_days = len(returns)
    years = total_days / 365.25

    # CAGR
    total_ret = equity.iloc[-1] / equity.iloc[0]
    cagr = total_ret ** (1 / years) - 1 if years > 0 else 0

    # Sharpe Ratio (annualized, daily returns)
    sharpe = returns.mean() / returns.std() * np.sqrt(365) if returns.std() > 0 else 0

    # Max Drawdown
    peak = equity.cummax()
    drawdown = (equity - peak) / peak
    max_dd = drawdown.min()

    # Win Rate (on days with non-zero returns while in position)
    active = returns[returns != 0]
    win_rate = (active > 0).mean() * 100 if len(active) > 0 else 0

    # Profit Factor
    gross_profit = active[active > 0].sum()
    gross_loss = abs(active[active < 0].sum())
    pf = gross_profit / gross_loss if gross_loss > 0 else np.inf

    return {
        "Label": label,
        "CAGR": f"{cagr:.1%}",
        "Sharpe": f"{sharpe:.2f}",
        "Max DD": f"{max_dd:.1%}",
        "Win Rate": f"{win_rate:.1f}%",
        "Profit Factor": f"{pf:.2f}",
        "Trades": trades,
        "Final Equity": f"${equity.iloc[-1] * 10000:,.0f}",
    }

# Compute for strategy and buy & hold
strat_metrics = compute_metrics(
    df["strat_ret"].dropna(),
    trades=int(df["trade"].sum()),
    label="SMA(50/200)"
)
bh_metrics = compute_metrics(
    df["ret"].dropna(),
    label="Buy & Hold"
)

results = pd.DataFrame([strat_metrics, bh_metrics]).set_index("Label")
results

## Part 4: Equity Curve + Drawdown Visualization

Two charts every backtest report needs:
1. **Equity curve** — strategy vs buy & hold
2. **Drawdown chart** — how deep and how long the losses get

In [ ]:
# Equity curves (starting at $10,000)
df["eq_strat"] = (1 + df["strat_ret"]).cumprod() * 10000
df["eq_bh"] = (1 + df["ret"]).cumprod() * 10000

# Drawdowns
df["dd_strat"] = df["eq_strat"] / df["eq_strat"].cummax() - 1
df["dd_bh"] = df["eq_bh"] / df["eq_bh"].cummax() - 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={"height_ratios": [2, 1]})

# --- Equity Curve ---
ax1.plot(df.index, df["eq_strat"], color="#3fb950", linewidth=1.5, label="SMA(50/200)")
ax1.plot(df.index, df["eq_bh"], color="#8b949e", linewidth=1, alpha=0.7, label="Buy & Hold")
ax1.fill_between(df.index, df["eq_strat"], df["eq_bh"],
                  where=df["eq_strat"] > df["eq_bh"], alpha=0.1, color="#3fb950")
ax1.fill_between(df.index, df["eq_strat"], df["eq_bh"],
                  where=df["eq_strat"] <= df["eq_bh"], alpha=0.1, color="#f85149")
ax1.set_ylabel("Equity ($)")
ax1.set_title("Equity Curve — SMA(50/200) vs Buy & Hold (No Costs)", fontsize=14)
ax1.legend(loc="upper left")
ax1.set_facecolor("#0d1117")

# --- Drawdown ---
ax2.fill_between(df.index, df["dd_strat"], 0, color="#f85149", alpha=0.4, label="Strategy DD")
ax2.plot(df.index, df["dd_bh"], color="#8b949e", linewidth=0.8, alpha=0.5, label="Buy & Hold DD")
ax2.set_ylabel("Drawdown")
ax2.set_title("Drawdown", fontsize=12)
ax2.legend(loc="lower left")
ax2.set_facecolor("#0d1117")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

fig.patch.set_facecolor("#0d1117")
plt.tight_layout()
plt.show()

## Part 5: Cost Sensitivity Analysis

**This is the most important part of any backtest.**

We test the same strategy with 4 different cost levels:
- **0 bp** — zero cost (what most backtests show you)
- **10 bp** — optimistic (maker fees only, no slippage)
- **18 bp** — realistic spot (7bp fee + 5bp slippage + 6bp spread)
- **50 bp** — conservative / perp futures with funding drag

> A strategy that only works at 0bp is **not a strategy** — it's an illusion.

**Key insight from our research (340 strategy variants tested):**
53% of published crypto backtests include zero transaction costs. When realistic costs are applied, most "profitable" strategies become unprofitable.

In [ ]:
cost_levels = {
    "0 bp (zero cost)": 0.0000,
    "10 bp (optimistic)": 0.0010,
    "18 bp (realistic spot)": 0.0018,
    "50 bp (conservative/perp)": 0.0050,
}

cost_results = []
equity_curves = {}

for label, cost_per_trade in cost_levels.items():
    # Subtract cost on each trade event
    cost_drag = df["trade"] * cost_per_trade
    net_ret = df["strat_ret"] - cost_drag
    eq = (1 + net_ret).cumprod() * 10000
    equity_curves[label] = eq

    # Metrics
    m = compute_metrics(net_ret.dropna(), trades=int(df["trade"].sum()), label=label)
    cost_results.append(m)

cost_df = pd.DataFrame(cost_results).set_index("Label")
cost_df

In [ ]:
# Visualize: how costs erode returns
colors = ["#3fb950", "#58a6ff", "#f0883e", "#f85149"]
fig, ax = plt.subplots(figsize=(14, 6))

for (label, eq), color in zip(equity_curves.items(), colors):
    ax.plot(df.index, eq, color=color, linewidth=1.5, label=label)

ax.plot(df.index, df["eq_bh"], color="#8b949e", linewidth=1, alpha=0.5, linestyle="--", label="Buy & Hold")
ax.set_ylabel("Equity ($)")
ax.set_title("Cost Sensitivity: Same Strategy, Different Cost Assumptions", fontsize=14)
ax.legend(loc="upper left", fontsize=10)
ax.set_facecolor("#0d1117")
fig.patch.set_facecolor("#0d1117")

# Annotate the gap
y_zero = equity_curves["0 bp (zero cost)"].iloc[-1]
y_real = equity_curves["18 bp (realistic spot)"].iloc[-1]
gap_pct = (y_zero - y_real) / y_zero * 100
ax.annotate(
    f"Cost drag: {gap_pct:.0f}% of\nzero-cost returns lost",
    xy=(df.index[-1], y_real), xytext=(-200, -60),
    textcoords="offset points", fontsize=11, color="#f0883e",
    arrowprops=dict(arrowstyle="->", color="#f0883e"),
)

plt.tight_layout()
plt.show()

print(f"\nThe difference between 0bp and 18bp = ${y_zero - y_real:,.0f} on a $10,000 account.")
print(f"That's {gap_pct:.0f}% of your 'profits' — gone to fees and slippage.")

---

## What This Notebook Shows vs. What It Doesn't

| This notebook (free) | Production bot (paid) |
|---|---|
| Basic SMA crossover | Custom strategies, multi-timeframe |
| Daily data only | 1m to 1d, any timeframe |
| No risk management | Capital Shield: stop loss, MDD limit, position sizing |
| No live execution | Auto-execute on Bybit/Binance |
| No alerts | Telegram alerts on every trade |
| No error handling | Retry logic, graceful degradation, logging |
| Manual analysis | Auto-generated HTML reports |

**This starter kit gives you the foundation. A production system needs 10x more engineering.**

---

### Need a production-grade crypto trading bot?

I build automated trading systems with **realistic backtests** (not the fairy-tale kind), risk management, and 24/7 execution.

**What makes me different:**
- SSRN-published researcher (340 strategy variants tested)
- Professional Engineer (Engineers Australia)
- Every backtest includes cost sensitivity analysis like Part 5 above

**[fiverr.com/jwquant](https://fiverr.com/jwquant)** — Message me before ordering.

---
*Trading involves risk. Past performance does not guarantee future results. This notebook is for educational purposes only.*